# 03. Pose Keypoint + MLP

MediaPipe 33 keypoints → 정규화 → 3-layer MLP 분류

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import json

from src import CLASSES
from src.dataset import PosturePoseDataset
from src.models import PoseMLP

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Step 1: MediaPipe 키포인트 추출
# 아직 추출 안 했다면 실행
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'src.pose_extract',
     '--img_root', 'data/raw',
     '--splits_csv', 'data/splits/splits.csv',
     '--output_csv', 'data/splits/pose_features.csv'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('[STDERR]', result.stderr[-500:])

In [ ]:
# 키포인트 특징 분포 탐색
df = pd.read_csv('data/splits/pose_features.csv')
print(f'전체 샘플: {len(df)}')
print(df['label'].value_counts())

feat_cols = [c for c in df.columns if c.startswith('kp_')]
print(f'특징 차원: {len(feat_cols)}')

In [ ]:
# 주요 keypoint 시각화 (어깨/목/엉덩이 y좌표)
# MediaPipe landmark 인덱스:
#   0=nose, 11=left_shoulder, 12=right_shoulder
#   23=left_hip, 24=right_hip, 7=left_ear

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, cls in zip(axes.flatten(), CLASSES):
    subset = df[df['label'] == cls]
    # 코 y좌표 (거북목 지표)
    nose_y = subset['kp_1']  # index 0*4+1 = 1
    # 왼쪽 어깨 y
    lshoulder_y = subset['kp_45']  # 11*4+1 = 45
    ax.scatter(nose_y, lshoulder_y, alpha=0.4, s=10)
    ax.set_title(cls)
    ax.set_xlabel('Nose Y (정규화)')
    ax.set_ylabel('Left Shoulder Y (정규화)')

plt.tight_layout()
plt.savefig('results/figures/keypoint_distribution.png', dpi=150)
plt.show()

In [ ]:
# MLP 학습
result = subprocess.run(
    [sys.executable, '-m', 'src.train_pose_mlp',
     '--epochs', '100',
     '--batch_size', '64',
     '--lr', '3e-4'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('[STDERR]', result.stderr[-1000:])

In [ ]:
# 학습 곡선
with open('results/metrics/pose_mlp_history.json') as f:
    history = json.load(f)

epochs = range(1, len(history['train_loss'])+1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ['loss', 'acc'], ['Loss', 'Accuracy']):
    ax.plot(epochs, history[f'train_{key}'], label='Train')
    ax.plot(epochs, history[f'val_{key}'], label='Val')
    ax.set_title(f'Pose+MLP {title}')
    ax.legend()
plt.tight_layout()
plt.show()